# DataMate: đo thời gian nạp model và thời gian dự đoán

Chạy trong container của image (thư mục `/app`), dữ liệu ở `/data`, kết quả ở `/result`. Notebook làm đúng các bước của `predict.sh`, in từng mốc thời gian.

In [ ]:
import json, os, subprocess, sys, time, urllib.request
DATA, OUT, MODEL, PORT = os.environ.get('DATA_DIR', '/data'), os.environ.get('OUT_DIR', '/result'), os.environ.get('MODEL', '/models/model.gguf'), 8091
os.makedirs(OUT, exist_ok=True)
os.chdir('/app')
t0 = time.time()
server = subprocess.Popen(['/opt/llama/llama-server', '-m', MODEL, '--jinja', '-ngl', '99', '-c', '8192', '--parallel', '1',
                           '--host', '127.0.0.1', '--port', str(PORT)],
                          stdout=open(f'{OUT}/llama_server.log', 'w'), stderr=subprocess.STDOUT)
ready = False
for _ in range(300):
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/health', timeout=2) as r:
            ready = json.load(r).get('status') == 'ok'
    except Exception:
        pass
    if ready or server.poll() is not None:
        break
    time.sleep(2)
t1 = time.time()
print(f'Model load time: {t1 - t0:.1f} s (ready={ready})')

In [ ]:
cmd = [sys.executable, '-m', 'datamate.predict', '--data', DATA, '--out', OUT]
if ready:
    cmd += ['--llm-url', f'http://127.0.0.1:{PORT}']
status = subprocess.call(cmd)
if status != 0 and ready:  # the LLM run failed: the rule-based reader still writes a file
    status = subprocess.call([sys.executable, '-m', 'datamate.predict', '--data', DATA, '--out', OUT])
t2 = time.time()
print(f'Prediction time: {t2 - t1:.1f} s')
print(f'Total time: {t2 - t0:.1f} s; exit status {status}')
server.terminate()

In [ ]:
lines = open(f'{OUT}/submission.jsonl', encoding='utf-8').read().splitlines()
print(len(lines), 'dòng trong submission.jsonl;', len(os.listdir(f'{OUT}/logs')), 'file log')